In [ ]:
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import os

def analyze_portfolio(csv_filepath):
    """
    Analyzes a stock portfolio from a CSV file.

    This function reads a CSV file with 'Ticker' and 'Shares' columns,
    fetches current stock prices and beta values (a measure of volatility),
    calculates the market value of each position, and generates a pie chart
    visualizing the portfolio composition alongside a data table.

    Args:
        csv_filepath (str): The path to the CSV file.
    """
    try:
        # Read the portfolio data from the provided CSV file (use the function parameter)
        portfolio_df = pd.read_csv(csv_filepath)
        print(f"Successfully loaded {csv_filepath}. Fetching stock data...")
        # Detect common column names for ticker and shares/position (case-insensitive common variants)
        possible_ticker_cols = ['TICKER','Ticker','ticker','SYMBOL','Symbol','symbol']
        possible_shares_cols = ['POSITION','Position','position','SHARES','Shares','shares','QTY','Qty','qty','QUANTITY','Quantity','quantity']
        ticker_col = next((c for c in portfolio_df.columns if c in possible_ticker_cols), None)
        shares_col = next((c for c in portfolio_df.columns if c in possible_shares_cols), None)
        if ticker_col is None or shares_col is None:
            print("Error: Could not find ticker or shares column in CSV. Available columns:", list(portfolio_df.columns))
            return
    except FileNotFoundError:
        print(f"Error: The file '{csv_filepath}' was not found.")
        print("Please ensure the CSV file is in the same directory as the script or provide the full path.")
        return

    # Lists to store the data fetched from yfinance
    prices = []
    market_values = []
    betas = []

    print(portfolio_df)

    # --- Data Fetching and Processing ---
    for index, row in portfolio_df.iterrows():
        ticker_symbol = row['TICKER']
        position_shares = row['POSITION']
        print(ticker_symbol)
        stock = yf.Ticker(ticker_symbol)
        info = stock.info
        print(info)
        # Get current price. Use 'regularMarketPrice' as a primary, with fallbacks.
        price = info.get('regularMarketPrice', info.get('currentPrice', 0))

            # Beta is a measure of a stock's volatility in relation to the overall market.
            # Beta > 1: More volatile than the market.
            # Beta < 1: Less volatile than the market.
            # A stock that swings more than the market over time has a beta above 1.0.
            # If less, it has a beta below 1.0.
        beta = info.get('beta', 1) # Beta might not be available for all tickers (e.g., mutual funds)

        prices.append(int(price))
        betas.append(beta)

            # Calculate the total market value of the position
        market_values.append(price * int(position_shares))

        print(f"  - Fetched data for {ticker_symbol}")

    # Add the new data to the DataFrame
    portfolio_df['Current Price'] = prices
    portfolio_df['Market Value'] = market_values
    portfolio_df['Beta (Risk)'] = betas

    # --- Calculate Total Portfolio Value ---
    total_portfolio_value = portfolio_df['Market Value'].sum()

    # --- Save to CSV ---
    # Save the analysis next to the input file so it's portable across environments
    output_dir = os.path.dirname(csv_filepath) or '.'
    output_csv_filepath = os.path.join(output_dir, 'portfolio_analysis.csv')
    portfolio_df.to_csv(output_csv_filepath, index=False)
    print(f"\nPortfolio analysis saved to {output_csv_filepath}")

    # --- Visualization ---
    print("\nGenerating portfolio visualization...")

    # Filter out any positions with zero or negative market value for a cleaner chart
    plot_df = portfolio_df[portfolio_df['Market Value'] > 0].copy()

    if plot_df.empty:
        print("\nNo valid portfolio data to display. Please check your ticker symbols.")
        return

    # Create the figure and axes for the plot
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 8))
    fig.suptitle('Portfolio Composition and Risk Analysis', fontsize=20)

    # 1. --- Pie Chart ---
    # Use a colormap based on Beta (Risk)
    # Normalize beta values for the colormap, handling None values
    valid_betas = plot_df['Beta (Risk)'].dropna()
    if not valid_betas.empty:
        norm = mcolors.Normalize(vmin=valid_betas.min(), vmax=valid_betas.max())
        cmap = cm.viridis
        colors = [cmap(norm(beta)) if beta is not None else 'grey' for beta in plot_df['Beta (Risk)']]
    else:
        # Default colors if no valid beta values are available
        colors = plt.cm.viridis(np.linspace(0.1, 0.9, len(plot_df)))


    wedges, texts, autotexts = ax1.pie(
        plot_df['Market Value'],
        labels=plot_df['TICKER'],
        autopct='%1.1f%%',
        startangle=140,
        pctdistance=0.85,
        colors=colors, # Use the beta-correlated colors
        wedgeprops=dict(width=0.4, edgecolor='w')
    )
    # Style the percentage text
    plt.setp(autotexts, size=10, weight="bold", color="black") # Changed color to black
    ax1.set_title('Portfolio Value Distribution by Risk (Beta)', fontsize=14) # Update title

    # Add a color bar for the beta values if valid betas exist
    if not valid_betas.empty:
        sm = cm.ScalarMappable(cmap=cmap, norm=norm)
        sm.set_array([])
        cbar = fig.colorbar(sm, ax=ax1, orientation='vertical', fraction=0.03, pad=0.05)
        cbar.set_label('Beta (Risk)', fontsize=12)


    # 2. --- Data Table ---
    ax2.axis('off') # Hide the axes for the table subplot

    # Format the data for display
    portfolio_df_display = portfolio_df.copy()
    # Remove the 'RISK' column if it exists; ignore if it doesn't to avoid KeyError
    portfolio_df_display = portfolio_df_display.drop(columns=['RISK'], errors='ignore')

    # Calculate and add Percentage column
    portfolio_df_display['Percentage'] = (portfolio_df_display['Market Value'] / total_portfolio_value) * 100
    # Convert Percentage to numeric for sorting
    portfolio_df_display['Percentage_numeric'] = portfolio_df_display['Percentage']

    # Sort by Percentage in descending order
    portfolio_df_display = portfolio_df_display.sort_values(by='Percentage_numeric', ascending=False)

    # Format Percentage column for display and drop the numeric column
    portfolio_df_display['Percentage'] = portfolio_df_display['Percentage_numeric'].map('{:.2f}%'.format)
    portfolio_df_display = portfolio_df_display.drop(columns=['Percentage_numeric'])

    portfolio_df_display['Current Price'] = portfolio_df_display['Current Price'].map('${:,.2f}'.format)
    portfolio_df_display['Market Value'] = portfolio_df_display['Market Value'].map('${:,.2f}'.format)
    # Handle missing beta values for display (use pandas notnull to catch NaN/None)
    portfolio_df_display['Beta (Risk)'] = portfolio_df_display['Beta (Risk)'].map(lambda b: f'{b:.2f}' if pd.notnull(b) else 'N/A')

    # Create the table
    table = ax2.table(
        cellText=portfolio_df_display.values,
        colLabels=portfolio_df_display.columns,
        loc='center',
        cellLoc='left',
        colWidths=[0.1, 0.1, 0.15, 0.15, 0.15, 0.15] # Adjusted colWidths to match the 6 columns + Percentage
    )
    table.auto_set_font_size(False)
    table.set_fontsize(11)
    table.scale(1.2, 1.2)

    # Style the table header
    for (i, j), cell in table.get_celld().items():
        if i == 0:
            cell.set_text_props(weight='bold', color='white')
            cell.set_facecolor('#40466e')

    ax2.set_title('Portfolio Data Breakdown', fontsize=14, y=0.85)

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()

if __name__ == '__main__':
    # The name of the CSV file containing the portfolio (must have 'Ticker' and 'Shares' columns)
    PORTFOLIO_FILE = 'Positions_main.csv'
    analyze_portfolio(PORTFOLIO_FILE)
    

Successfully loaded Positions_main.csv. Fetching stock data...
  TICKER POSITION
0   AMZN        6
1   GOOG       10
2   ONDS      100
3    NaN        Í
AMZN
{'address1': '410 Terry Avenue North', 'city': 'Seattle', 'state': 'WA', 'zip': '98109-5210', 'country': 'United States', 'phone': '206 266 1000', 'website': 'https://www.amazon.com', 'industry': 'Internet Retail', 'industryKey': 'internet-retail', 'industryDisp': 'Internet Retail', 'sector': 'Consumer Cyclical', 'sectorKey': 'consumer-cyclical', 'sectorDisp': 'Consumer Cyclical', 'longBusinessSummary': "Amazon.com, Inc. engages in the retail sale of consumer products, advertising, and subscriptions service through online and physical stores in North America and internationally. The company operates through three segments: North America, International, and Amazon Web Services (AWS). It also manufactures and sells electronic devices, including Kindle, fire tablets, fire TVs, echo, ring, blink, and eero; and develops and produces me

AttributeError: 'float' object has no attribute 'upper'